# 80문항 파이프라인 검수 (자동 실행 → 검수 카드 → 사람 판정)

`storage/test_data` 에 넣은 평가 질문 파일(자동 탐색, csv/jsonl/json/xlsx)을 **새 파이프라인**(`POST /api/v1/chat` 과 같은 `run_chat`: ① 재작성 → ② 검색·View → ③ 답변)으로 돌리고,
문항마다 **JSON + 검수 카드(HTML/PNG)** 를 만든 뒤 **사람이 지표·최종 판정·코멘트를 입력**합니다.
(구 `eval/golden_set/kosis_fast_human_review_accum.ipynb` 와 같은 방식, 같은 JSON 형식)

| 단계 | 셀 | 결과 (`storage/eval_runs/<질문 파일 이름>/`) |
|---|---|---|
| Phase 1 자동 실행 | Cell 2 | `auto_results.json`, `json/Q001.json`, `cards/Q001.html·png`, `human_final.csv` |
| Phase 2 요약 | Cell 3 | 신호등 요약표 |
| Phase 2 사람 판정 | Cell 4 | QID 바꿔 가며 카드 보고 판정 → 위 파일 모두 갱신 |
| (대안) Streamlit | — | `streamlit run eval/review_app.py` → 같은 파일을 버튼으로 검수 |

- 자동 지표 5개는 **초안**입니다. 최종 판정은 사람이 정하고, 다시 실행해도 사람이 검수한 문항(`reviewed=true`)의 판정은 유지됩니다
- 사전 준비: `docs/PIPELINE.md` 의 A1·A2 (메타 프로필 + 표 색인), `.env` 의 HCX 키

In [ ]:
# [Cell 1] 환경 설정
import importlib
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").is_dir())   # 저장소 루트
for p in (ROOT, ROOT / "eval" / "pipeline_review"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import review_80 as rv
importlib.reload(rv)
from IPython.display import HTML, display

from backend.core.config import TEST_DATA_DIR

rv.list_sets()                        # storage/test_data 에서 찾은 테스트 세트 (번호·문항 수·열 매핑·진행 현황)

QUERIES = "80"        # 테스트 세트: 번호 | 파일 이름 일부 | 경로 (하나뿐이면 None)
RUN = rv.run_for(QUERIES)             # 결과 폴더 = storage/eval_runs/<질문 파일 이름>  (설정 비교용으로 따로 두려면 rv.RunPaths("이름"))
QUESTIONS = rv.load_questions(QUERIES)
print(f"\n선택: {len(QUESTIONS)}문항 ← {TEST_DATA_DIR}")
print(f"정답 참고 CSV 결합: {sum(bool(q['reference']) for q in QUESTIONS)}문항")
print(f"결과 폴더 → {RUN.dir}")

---
## [Phase 1] 자동 실행 — 질문마다 ① 재작성 → ② 검색·View → ③ 답변 + 5대 지표 자동 채점
- HCX 호출: 문항당 재작성 1회 + 답변 1회 + 임베딩(캐시 없을 때)
- 매 문항 저장 → 중간에 멈춰도 다시 실행하면 **안 끝난 문항만** 이어서 실행 (`[API 호출 오류]` 문항은 다시 시도)
- 처음에는 `N = 5` 로 확인 후 `N = None` 으로 전체

In [ ]:
# [Cell 2] Phase 1 자동 실행
IDS = None            # 예: ["Q001", "Q014"]   (None = 전체)
N = 5                 # 앞에서 N문항만 (None = 전체)
FORCE = False         # True = 이미 실행한 문항도 다시 (사람 판정은 유지)
TODAY = "2026-11-02"  # 평가셋 기준일 (작년·최근 N년 해석)

_ = rv.run_batch(RUN, ids=IDS, n=N, force=FORCE, today=TODAY, top_k=10, n_views=3, png=True, queries=QUERIES)

---
## [Phase 2] 요약표 — 어느 문항이 PARTIAL / FAIL 인지 한눈에
열: 검색(검색_적합성) · 원본(원본_수치_추출) · Context(Context_충분성) · 변화율(변화율_응답_완성도) · 자동(자동_환각_검증) = **현재 판정**(사람이 고쳤으면 그 값), `자동` 열 = 자동 종합 판정

In [ ]:
# [Cell 3] 신호등 요약표
df = rv.summary_df(RUN)
COLORS = {"PASS": "background-color:#d4edda;color:#155724", "PARTIAL": "background-color:#fff3cd;color:#856404",
          "FAIL": "background-color:#f8d7da;color:#721c24"}
sty = df.style
display((sty.map if hasattr(sty, "map") else sty.applymap)(lambda v: COLORS.get(v, "")))
print("최종:", df["최종"].value_counts().to_dict(), "| 검수 완료", int((df["검수"] == "✅").sum()), "/", len(df))

---
## [Phase 2] 검수 카드 + 사람 판정
1. `QID` 만 바꾸고 실행 → 카드 확인 (질문 · ①재작성/②View 조건 · 답변(미검증 숫자 빨간 표시) · 5대 지표 · 검색 후보 · 근거 View 표)
2. 판정을 정했으면 `FINAL_VERDICT` / `COMMENT_TEXT` / `DETAIL` 을 채우고 다시 실행 → `auto_results.json`, `json/QID.json`, `human_final.csv`, `cards/QID.html·png` 모두 갱신
3. 다음 문항으로 넘어가기 전에 `FINAL_VERDICT`, `COMMENT_TEXT` 를 `None` 으로 되돌리면 실수로 덮어쓰지 않음

In [ ]:
# [Cell 4] 카드 보기 + 판정 입력 (QID 바꿔 가며 Ctrl+Enter)
QID = "Q001"

FINAL_VERDICT = None   # "PASS" | "PARTIAL" | "FAIL" | None (그대로)
COMMENT_TEXT = None    # "코멘트" | None (그대로)
DETAIL = {             # 세부 지표를 고칠 때만 (None = 그대로)
    "검색_적합성": None,
    "원본_수치_추출": None,
    "Context_충분성": None,
    "변화율_응답_완성도": None,
    "자동_환각_검증": None,
}

if FINAL_VERDICT or COMMENT_TEXT is not None or any(DETAIL.values()):
    rec = rv.set_verdict(RUN, QID, final=FINAL_VERDICT, comment=COMMENT_TEXT, **DETAIL)
    print(f"💾 {QID} 저장 → {RUN.final_csv.name}, json/{QID}.json, cards/{QID}.html")
else:
    rec = rv.load_results(RUN)[QID]

display(HTML(rv.card_html(rec)))
png = RUN.card_dir / f"{QID}.png"
if not png.exists():
    rv.write_card(RUN, rec, png=True)      # PNG 가 없으면 지금 만듦
print("PNG:", png if png.exists() else "없음 → pip install html2image (Chrome/Edge 필요) 또는 pip install playwright && playwright install chromium")

---
### 참고
- **전체 카드 PNG 다시 만들기**: 아래 셀, 또는 `python eval/pipeline_review/review_80.py cards --png`
- **Streamlit 으로 검수**: `streamlit run eval/review_app.py` — 가장 최근 `storage/eval_runs/*/auto_results.json` 을 자동으로 엶. 라디오 버튼으로 판정하면 같은 파일들이 갱신됨
- **CLI 로 실행**: `python eval/pipeline_review/review_80.py run --n 5`, `... summary`
- 자동 지표 기준은 `review_80.py` 의 `score()` 에 있습니다 (정답 참고 CSV 에 `tbl` 이 들어간 열이 있으면 표 ID 로, 없으면 조사명으로 검색 적합성 판단)

In [ ]:
# [Cell 5] (선택) 모든 카드 PNG 다시 만들기
rv.write_cards(RUN, png=True)